# AREA: fine-tuning Laya-multilingual on the MIT causal taxonomy

Fine-tunes [`convaiinnovations/laya-multilingual`](https://huggingface.co/convaiinnovations/laya-multilingual)
on the causal labels (entity, intent, timing) of the MIT AI Risk Repository with the scripts of the
[AREA repository](https://github.com/6MicheleStingo9/AREA/tree/rework/area) (`training/`), then evaluates it
on the MIT eval split and on the AREA risks with `training/evaluate_laya.py`.

**Settings:** Accelerator *GPU T4 x2* (or P100), Internet *on*, then *Run all*.

**Output** (Output panel, `/kaggle/working/`): `laya-causal.zip` (the checkpoint, about 0.7 GB) and
`results/` (metrics and predictions).

In [ ]:
%env USE_TF=0
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone --depth 1 -b rework/area https://github.com/6MicheleStingo9/AREA.git
%cd AREA
!git log -1 --format='%h %s'
!pip install -q laya==0.4.1

Training and test files in Laya's format (`training/data/`): one row per risk, three typed questions.

In [ ]:
!python -m training.build_laya_data

Fine-tuning: 4 epochs, RLCD objective, temperatures fitted on a held-out calibration slice.

In [ ]:
!python -m training.finetune_laya --device cuda

Evaluation on the MIT eval split and the AREA risks (same metrics as the Gemini baseline, plus calibration).

In [ ]:
!python -m training.evaluate_laya --model training/checkpoints/laya-causal --name laya-causal

Package the checkpoint (without the per-epoch copy) and the results for download.

In [ ]:
!mkdir -p /kaggle/working/results && cp evaluation/results/*laya-causal* /kaggle/working/results/
!cd training/checkpoints && zip -q -r /kaggle/working/laya-causal.zip laya-causal -x 'laya-causal/checkpoint_latest/*'
!ls -lh /kaggle/working /kaggle/working/results